# Análisis de Intentos de Suicidio — SIVIGILA Colombia 2024
## Proyecto Final · Bootcamp Talento Tech – Inteligencia Artificial

**Evento SIVIGILA:** 356 — Intento de suicidio  
**Período:** Año 2024  
**Metodología:** CRISP-DM (Cross-Industry Standard Process for Data Mining)  
**Enfoque:** Preventivo, territorial y de salud pública  

---

### Objetivo del proyecto

Construir una plataforma de inteligencia de datos que permita:
1. Comprender los patrones epidemiológicos de los intentos de suicidio en Colombia durante 2024.
2. Identificar municipios con mayor necesidad de intervención preventiva.
3. Aplicar modelos de IA descriptivos, predictivos y prescriptivos.
4. Generar recomendaciones basadas en evidencia para apoyar la toma de decisiones en salud pública.

> **Nota ética:** Este análisis es de carácter epidemiológico y poblacional. No realiza diagnósticos clínicos ni identifica individuos. Todo el enfoque es preventivo.

---
## Fase 1 — Comprensión del negocio (Business Understanding)

### Contexto
El SIVIGILA (Sistema Nacional de Vigilancia en Salud Pública) recopila los eventos de interés en salud pública notificados por las Unidades Primarias Generadoras de Datos (UPGD) en todo el territorio colombiano. El Evento 356 corresponde a los intentos de suicidio.

### Pregunta de investigación
> ¿En qué municipios debe concentrarse la intervención preventiva en salud mental, y qué características demográficas y territoriales explican mayor riesgo?

### Productos esperados
- Dataset limpio y documentado
- Análisis descriptivo por territorio, edad, sexo y temporalidad
- Índice de Prioridad de Intervención (IPI) por municipio
- Modelos supervisados: Árbol de Decisión + Random Forest
- Modelo no supervisado: K-Means
- Dashboard interactivo en Streamlit
- Recomendaciones prescriptivas

---
## Fase 2 — Comprensión de los datos (Data Understanding)

In [ ]:
# ============================================================
# CELDA 1: Importaciones y configuración
# Todas las librerías necesarias para el análisis completo.
# pandas y numpy son las bases del análisis de datos.
# matplotlib y seaborn para visualizaciones estáticas.
# sklearn para los modelos de IA.
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
import sys
import os

warnings.filterwarnings('ignore')

# Agregar la raíz del proyecto al path para importar src/
sys.path.insert(0, os.path.dirname(os.path.abspath('')))

# Paleta visual: Celadon y Azul Bondi (sobria para salud pública)
PRIMARIO    = '#0D7C8F'   # Azul Bondi
SECUNDARIO  = '#A8D5BA'   # Celadon
ACENTO      = '#5BA4B0'   # Azul Bondi medio
ADVERTENCIA = '#E07B54'   # Terracota suave
FONDO       = '#F7FBFA'   # Fondo general

plt.rcParams.update({
    'figure.facecolor': FONDO,
    'axes.facecolor':   FONDO,
    'axes.titlesize':   13,
    'axes.titleweight': 'bold',
})

print('✓ Librerías importadas correctamente')
print(f'  pandas  : {pd.__version__}')
print(f'  numpy   : {np.__version__}')

In [ ]:
# ============================================================
# CELDA 2: Carga del archivo SIVIGILA
# Se leen las columnas de código geográfico como texto (str)
# para evitar que pandas elimine los ceros a la izquierda
# del código DANE (por ejemplo, '05' → 5 sería incorrecto).
# ============================================================

RUTA_RAW = '../data/raw/suicidios_2024.xlsx'

df_raw = pd.read_excel(
    RUTA_RAW,
    sheet_name='Hoja1',
    dtype={
        'COD_DPTO_O': str,
        'COD_MUN_O':  str,
    }
)

print(f'Filas    : {df_raw.shape[0]:,}')
print(f'Columnas : {df_raw.shape[1]}')
print(f'Evento   : {df_raw["Nombre_evento"].iloc[0]}')
print(f'Año      : {df_raw["ANO"].iloc[0]}')
df_raw.head(3)

In [ ]:
# ============================================================
# CELDA 3: Perfil de calidad de datos
# Antes de limpiar, documentamos el estado inicial.
# Esto es parte del 'Data Understanding' en CRISP-DM.
# ============================================================

total = len(df_raw)
print('=== CALIDAD DE DATOS ===')
print(f'Filas duplicadas       : {df_raw.duplicated().sum()}')
print(f'CONSECUTIVE duplicado  : {df_raw["CONSECUTIVE"].duplicated().sum()}')
print()

# Porcentaje de nulos por columna (solo las que tienen > 0 %)
nulos = (df_raw.isna().sum() / total * 100).round(1)
nulos_sig = nulos[nulos > 0].sort_values(ascending=False)
print('Columnas con valores nulos > 0 %:')
print(nulos_sig.to_string())

print()
print('=== ESTADÍSTICAS DESCRIPTIVAS – columnas clave ===')
df_raw[['EDAD', 'SEMANA']].describe().round(2)

In [ ]:
# ============================================================
# CELDA 4: Distribuciones iniciales de variables clave
# ============================================================

print('SEXO:')
print(df_raw['SEXO'].value_counts(dropna=False).to_string())

print('\nÁREA (1=Cabecera, 2=Centro poblado, 3=Rural):')
print(df_raw['AREA'].value_counts(dropna=False).to_string())

print('\nPAC_HOS (1=Hospitalizado, 2=No hospitalizado):')
print(df_raw['PAC_HOS'].value_counts(dropna=False).to_string())

print('\nTop 5 departamentos de ocurrencia:')
print(df_raw['Departamento_ocurrencia'].value_counts().head(5).to_string())

### Hallazgos del Data Understanding

| Variable | Hallazgo |
|---|---|
| Filas totales | **38.769** casos confirmados por clínica |
| Duplicados | **0** — dataset íntegro |
| Columnas con nulos > 5% | `GRU_POB`, `FEC_DEF`, `CBMTE`, `CER_DEF`, `FM_*`, `FEC_HOS` |
| SEXO | 63.9% Femenino, 36.1% Masculino |
| EDAD mediana | 22 años |
| Adolescentes (12-17) | 27.9% del total |
| Departamentos | 34 reportan casos |
| Municipios | 1.049 únicos |

**Decisión clave:** Las columnas `GRU_POB`, `FEC_DEF`, `CBMTE`, `CER_DEF` y militares (`FM_*`) tienen ≥99% de nulos y serán descartadas. Esto es esperado: la base solo contiene intentos (no fallecidos), y los registros militares son mínimos.

---
## Fase 3 — Preparación de los datos (Data Preparation)

In [ ]:
# ============================================================
# CELDA 5: Pipeline de limpieza
# Se importa desde src/limpieza.py para evitar duplicar código.
# Cada paso está documentado en ese archivo con su justificación.
# ============================================================

from src.limpieza import limpiar_datos, guardar_procesado

df = limpiar_datos(verbose=True)

In [ ]:
# ============================================================
# CELDA 6: Verificación del dataset limpio
# Revisamos las nuevas columnas derivadas.
# ============================================================

print('Nuevas columnas derivadas:')
nuevas = ['cod_dane_mun', 'mes', 'mes_nombre', 'trimestre', 'semestre',
          'grupo_edad', 'sexo_nombre', 'area_nombre', 'hospitalizado',
          'es_menor', 'es_adolescente', 'es_rural']
print(df[nuevas].head(5).to_string())

print(f'\nColumnas finales : {df.shape[1]}')
print(f'Filas finales    : {len(df):,}')

In [ ]:
# ============================================================
# CELDA 7: Código DANE de 5 dígitos — por qué es crítico
# COD_MUN_O solo no es único entre departamentos.
# Ejemplo: municipio código '001' existe en TODOS los deptos
# (siempre es la capital). La clave correcta es DPTO+MUN.
# ============================================================

print('Municipios únicos con COD_MUN_O solo:', df['COD_MUN_O'].nunique())
print('Municipios únicos con cod_dane_mun (5 dígitos):', df['cod_dane_mun'].nunique())
print()
print('Ejemplo — municipios con código 001 (serían la misma si no usáramos DPTO):')
print(df[df['COD_MUN_O']=='001'][['COD_DPTO_O','Departamento_ocurrencia',
                                   'cod_dane_mun','Municipio_ocurrencia']]
      .drop_duplicates().head(8).to_string(index=False))

In [ ]:
# ============================================================
# CELDA 8: Detección de outliers en EDAD
# Usamos el método IQR (rango intercuartílico).
# Importante: en salud pública NO se eliminan los extremos.
# Un niño de 5 años o un adulto de 90 son casos clínicamente
# válidos que deben permanecer en el análisis.
# ============================================================

Q1  = df['EDAD'].quantile(0.25)
Q3  = df['EDAD'].quantile(0.75)
IQR = Q3 - Q1
lim_inf = Q1 - 1.5 * IQR
lim_sup = Q3 + 1.5 * IQR

outliers = df[(df['EDAD'] < lim_inf) | (df['EDAD'] > lim_sup)]

print(f'Q1 = {Q1:.0f}, Q3 = {Q3:.0f}, IQR = {IQR:.0f}')
print(f'Límite inferior : {lim_inf:.1f}')
print(f'Límite superior : {lim_sup:.1f}')
print(f'Casos fuera de rango : {len(outliers):,} ({len(outliers)/len(df)*100:.1f}%)')
print()
print('Decisión: SE CONSERVAN todos los registros.')
print('Justificación: los valores extremos de edad son clínicamente válidos.')

---
## Fase 4 — Análisis Exploratorio (EDA)

In [ ]:
# ============================================================
# CELDA 9: KPIs globales
# Resumen ejecutivo del dataset.
# ============================================================

from src.eda import resumen_general
kpis = resumen_general(df)

print('=' * 50)
print('  RESUMEN EJECUTIVO — SIVIGILA 2024')
print('=' * 50)
for k, v in kpis.items():
    print(f'  {k:<30s}: {v}')

In [ ]:
# ============================================================
# CELDA 10: Distribución por sexo
# Resultado: 63.9% Femenino vs 36.1% Masculino.
# Esto es consistente con literatura internacional:
# las mujeres tienen mayor frecuencia de intentos;
# los hombres mayor letalidad (pero esta base solo
# registra intentos, no suicidios consumados).
# ============================================================

from src.eda import casos_por_sexo
tabla_sexo = casos_por_sexo(df)
print(tabla_sexo.to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

axes[0].bar(tabla_sexo['sexo_nombre'], tabla_sexo['total_casos'],
            color=[PRIMARIO, SECUNDARIO], edgecolor='white')
axes[0].set_title('Casos por sexo (n)')
axes[0].set_ylabel('Número de casos')
for i, v in enumerate(tabla_sexo['total_casos']):
    axes[0].text(i, v + 100, f'{v:,}', ha='center')

axes[1].pie(tabla_sexo['total_casos'],
            labels=tabla_sexo['sexo_nombre'],
            autopct='%1.1f%%',
            colors=[PRIMARIO, SECUNDARIO],
            startangle=90,
            wedgeprops=dict(width=0.55, edgecolor='white'))
axes[1].set_title('Proporción por sexo (%)')

fig.suptitle('Distribución por sexo — Intentos de suicidio Colombia 2024',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../outputs/graficas/nb_01_sexo.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ============================================================
# CELDA 11: Distribución por grupo de edad
# HALLAZGO CLAVE: el grupo 18–25 es el más numeroso (31.0%),
# pero el grupo 12–17 (adolescentes) concentra el 27.9%.
# Juntos, los menores de 25 años representan el 60.8%.
# Esto señala una problemática prioritaria en población joven.
# ============================================================

from src.eda import casos_por_grupo_edad
tabla_edad = casos_por_grupo_edad(df)
print(tabla_edad.to_string(index=False))

orden = ['0–11', '12–17', '18–25', '26–35', '36–45', '46–59', '60+']
tabla_edad['grupo_edad'] = pd.Categorical(
    tabla_edad['grupo_edad'].astype(str),
    categories=orden, ordered=True
)
tabla_edad = tabla_edad.sort_values('grupo_edad')

colores = [ADVERTENCIA if g in ['12–17','18–25'] else PRIMARIO
           for g in tabla_edad['grupo_edad'].astype(str)]

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(tabla_edad['grupo_edad'].astype(str),
              tabla_edad['total_casos'],
              color=colores, edgecolor='white')
for bar in bars:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 50,
            f'{int(bar.get_height()):,}', ha='center', fontsize=9)

ax.set_title('Distribución por grupo de edad\nIntentos de suicidio — Colombia 2024')
ax.set_xlabel('Grupo de edad')
ax.set_ylabel('Número de casos')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))

# Anotación interpretativa
ax.annotate('60.8% tiene\nmenos de 26 años',
            xy=(2, 12011), xytext=(4, 11000),
            arrowprops=dict(arrowstyle='->', color=ADVERTENCIA),
            color=ADVERTENCIA, fontsize=9, ha='center')

plt.tight_layout()
plt.savefig('../outputs/graficas/nb_02_grupo_edad.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ============================================================
# CELDA 12: Tendencia temporal por mes y semana
# HALLAZGO: el mes de mayor registro es septiembre (3.755 casos).
# El primer semestre tuvo 18.140 casos y el segundo 20.629.
# Esto representa un incremento del 13.7% en el segundo semestre.
# ============================================================

from src.eda import casos_por_mes, tendencia_semanal

tabla_mes = casos_por_mes(df)
tabla_sem = tendencia_semanal(df)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 8))

# Por mes
ax1.bar(tabla_mes['mes_nombre'], tabla_mes['total_casos'],
        color=PRIMARIO, edgecolor='white', alpha=0.85)
ax1.set_title('Casos por mes de notificación')
ax1.set_ylabel('Casos')
ax1.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))

# Por semana con media móvil
ax2.plot(tabla_sem['SEMANA'], tabla_sem['casos'],
         color=ACENTO, linewidth=1, alpha=0.6, label='Casos por semana')
ax2.plot(tabla_sem['SEMANA'], tabla_sem['media_movil_4sem'],
         color=PRIMARIO, linewidth=2.5, label='Media móvil 4 semanas')
ax2.fill_between(tabla_sem['SEMANA'], tabla_sem['casos'],
                 alpha=0.1, color=PRIMARIO)
ax2.set_title('Tendencia semanal epidemiológica')
ax2.set_xlabel('Semana epidemiológica')
ax2.set_ylabel('Casos')
ax2.legend()
ax2.set_xlim(1, 52)

# Marcar H1 vs H2
ax2.axvline(26, color=ADVERTENCIA, linestyle='--', alpha=0.7)
ax2.text(13, tabla_sem['casos'].max()*0.95, 'H1', ha='center',
         color=ADVERTENCIA, fontweight='bold')
ax2.text(39, tabla_sem['casos'].max()*0.95, 'H2 (+13.7%)', ha='center',
         color=ADVERTENCIA, fontweight='bold')

fig.suptitle('Evolución temporal — Intentos de suicidio Colombia 2024',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../outputs/graficas/nb_03_tendencia.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ============================================================
# CELDA 13: Top 15 departamentos
# Antioquia y Bogotá concentran el 31% de los casos.
# Pero esto se explica en parte por su mayor población.
# El IPI (Fase 5) corrige esto usando tasas por 100k hab.
# ============================================================

from src.eda import casos_por_departamento
tabla_depto = casos_por_departamento(df, top=15)
print(tabla_depto.to_string(index=False))

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.barh(tabla_depto['Departamento_ocurrencia'],
               tabla_depto['total_casos'],
               color=PRIMARIO, edgecolor='white')
for bar in bars:
    ax.text(bar.get_width() + 30, bar.get_y() + bar.get_height()/2,
            f'{int(bar.get_width()):,}', va='center', fontsize=8)
ax.invert_yaxis()
ax.set_title('Top 15 departamentos por número de casos\nIntentos de suicidio — Colombia 2024')
ax.set_xlabel('Número de casos')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
plt.tight_layout()
plt.savefig('../outputs/graficas/nb_04_departamentos.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ============================================================
# CELDA 14: Mapa de calor — mes × sexo
# Permite identificar si hay estacionalidad diferencial por sexo.
# ============================================================

from src.eda import cruce_mes_sexo
tabla_heat = cruce_mes_sexo(df)
print(tabla_heat)

fig, ax = plt.subplots(figsize=(7, 7))
sns.heatmap(
    tabla_heat,
    annot=True, fmt=',d',
    cmap=sns.light_palette(PRIMARIO, as_cmap=True),
    linewidths=0.5,
    ax=ax
)
ax.set_title('Mapa de calor: mes × sexo\nIntentos de suicidio — Colombia 2024')
ax.set_xlabel('Sexo')
ax.set_ylabel('Mes')
plt.tight_layout()
plt.savefig('../outputs/graficas/nb_05_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ============================================================
# CELDA 15: Cruce sexo × grupo de edad
# Tabla cruzada (crosstab) para ver la distribución conjunta.
# ============================================================

from src.eda import cruce_sexo_grupo_edad
cruce = cruce_sexo_grupo_edad(df)
print(cruce)

# Interpretación
print('\nInterpretación:')
print('El grupo 12-17 Femenino es el subgrupo de mayor volumen absoluto.')
print('Esto es consistente con la literatura sobre conducta suicida en adolescentes.')

In [ ]:
# ============================================================
# CELDA 16: Boxplot — distribución de edad por sexo
# Permite comparar la dispersión de edades entre sexos.
# ============================================================

fig, ax = plt.subplots(figsize=(8, 5))
data_f = df[df['SEXO'] == 'F']['EDAD'].dropna()
data_m = df[df['SEXO'] == 'M']['EDAD'].dropna()

bp = ax.boxplot(
    [data_f, data_m],
    labels=['Femenino', 'Masculino'],
    patch_artist=True,
    medianprops=dict(color='white', linewidth=2)
)
bp['boxes'][0].set_facecolor(PRIMARIO)
bp['boxes'][1].set_facecolor(SECUNDARIO)

ax.set_title('Distribución de edad por sexo')
ax.set_ylabel('Edad (años)')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

print(f'Mediana Femenino  : {data_f.median():.0f} años')
print(f'Mediana Masculino : {data_m.median():.0f} años')

In [ ]:
# ============================================================
# CELDA 17: Grupos poblacionales de riesgo (GP_*)
# Banderas binarias de SIVIGILA: 1 = pertenece al grupo.
# GP_PSIQUIA (antecedente psiquiátrico) es el más relevante:
# 1.338 casos (3.45%), lo que es un factor de riesgo establecido.
# ============================================================

from src.eda import perfil_grupos_riesgo
tabla_gp = perfil_grupos_riesgo(df)
print(tabla_gp[tabla_gp['grupo'] != 'GP_OTROS'].to_string(index=False))

### Resumen del EDA — Hallazgos principales

| # | Hallazgo | Implicación |
|---|---|---|
| 1 | **38.769 casos** en 2024, 0 duplicados | Base íntegra y confiable |
| 2 | **63.9%** de los casos son mujeres | Estrategias con enfoque de género |
| 3 | **27.9%** son adolescentes (12–17) | Prioridad en entornos escolares |
| 4 | **60.8%** tiene menos de 26 años | Foco en salud mental joven |
| 5 | Pico en **septiembre** | Revisar factores de temporalidad |
| 6 | **H2 creció 13.7%** vs H1 | Tendencia ascendente en el año |
| 7 | **64.7%** fueron hospitalizados | Alta severidad clínica general |
| 8 | **10%** en zona rural | Barreras de acceso en territorios dispersos |

---
## Fase 5 — Sistema de Priorización Territorial (IPI)

### ¿Por qué no basta con el conteo absoluto de casos?

Bogotá registra **5.974 casos** y Mitú (Vaupés) solo **75 casos**. Sin embargo, Mitú tiene una tasa de **227 casos por cada 100.000 habitantes**, mientras Bogotá tiene ~71. Usar solo conteos absolutos invisibiliza municipios pequeños con altísimo riesgo relativo.

### Fórmula del IPI

```
IPI = 0.35 × tasa_x100k           (carga epidemiológica relativa)
    + 0.25 × pct_adolescentes      (alerta de población vulnerable)
    + 0.20 × tendencia_H2_H1       (velocidad de crecimiento)
    + 0.10 × pct_hospit            (proxy de severidad clínica)
    + 0.10 × pct_psiquia           (factor de riesgo documentado)
```

Todas las variables se normalizan **min-max** a [0,1] antes de aplicar los pesos. El resultado se escala a [0, 100].

In [ ]:
# ============================================================
# CELDA 18: Construcción del IPI
# ============================================================

from src.ipi import pipeline_ipi, municipios_emergentes

mun = pipeline_ipi(df, verbose=True)

In [ ]:
# ============================================================
# CELDA 19: Visualización del IPI — distribución y ranking
# ============================================================

# Distribución de niveles de prioridad
dist = mun['nivel_prioridad'].value_counts()
print('Distribución de prioridades:')
print(dist.to_string())

# Top 10 por IPI
print('\nTop 10 municipios por IPI:')
top10 = mun[['ranking_nacional','Departamento_ocurrencia','Municipio_ocurrencia',
              'total_casos','tasa_x100k','IPI','nivel_prioridad']].head(10)
print(top10.to_string(index=False))

In [ ]:
# ============================================================
# CELDA 20: Scatter — tasa vs % adolescentes (tamaño = IPI)
# Cada punto es un municipio. El tamaño refleja el IPI.
# Municipios arriba a la derecha: mayor carga + mayor vulnerabilidad.
# ============================================================

mun_plot = mun[mun['total_casos'] >= 5].copy()

colores_scatter = {
    'Crítica': '#C0392B', 'Alta': '#E07B54',
    'Media': '#5BA4B0',   'Baja': '#A8D5BA'
}

fig, ax = plt.subplots(figsize=(11, 6))
for nivel, grupo in mun_plot.groupby('nivel_prioridad'):
    ax.scatter(
        grupo['tasa_x100k'],
        grupo['pct_adolescentes'] * 100,
        s=grupo['IPI'] * 3,
        c=colores_scatter.get(nivel, ACENTO),
        alpha=0.65,
        edgecolors='white',
        linewidths=0.5,
        label=nivel
    )

# Etiquetar los 5 primeros
for _, row in mun.head(5).iterrows():
    if row['total_casos'] >= 5:
        ax.annotate(row['Municipio_ocurrencia'],
                    (row['tasa_x100k'], row['pct_adolescentes']*100),
                    fontsize=7, color='#333')

ax.set_title('Tasa por 100k hab. vs % Adolescentes (tamaño = IPI)\nMunicipios con ≥5 casos')
ax.set_xlabel('Tasa por 100.000 habitantes')
ax.set_ylabel('% Adolescentes (12–17)')
ax.legend(title='Nivel IPI')
plt.tight_layout()
plt.savefig('../outputs/graficas/nb_06_scatter_ipi.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ============================================================
# CELDA 21: Municipios emergentes
# Municipios con crecimiento acelerado en H2 (>50%) pero
# que aún no están en prioridad Crítica.
# Son los de mayor riesgo de escalar en 2025.
# ============================================================

emerg = municipios_emergentes(mun, top=10)
print('Municipios con crecimiento acelerado (emergentes):')
print(emerg[['Departamento_ocurrencia','Municipio_ocurrencia',
             'total_casos','tendencia_H2_H1','IPI','nivel_prioridad']]
      .to_string(index=False))

---
## Fase 6 — Modelado (Modeling)

### Estrategia de modelado

La unidad de análisis es el **municipio** (no el caso individual). Para cada municipio con ≥5 casos, calculamos sus indicadores agregados. La variable objetivo es `nivel_prioridad` (Baja / Media / Alta / Crítica).

**¿Por qué no modelar a nivel de caso individual?**  
Toda fila de SIVIGILA ya es un intento confirmado, no hay un "ocurrió / no ocurrió" que predecir. El análisis predictivo tiene sentido a nivel territorial: clasificar qué municipios requieren intervención.

**Tres modelos complementarios:**
- **Árbol de Decisión:** interpretabilidad máxima, reglas legibles
- **Random Forest:** mejor rendimiento, feature importance
- **K-Means:** agrupamiento sin variable objetivo, perfiles territoriales

In [ ]:
# ============================================================
# CELDA 22: Preparación de datos para el modelo
# Solo municipios con ≥5 casos para robustez estadística.
# Variables seleccionadas: todas las del IPI más indicadores
# adicionales disponibles en la tabla municipal.
# ============================================================

from src.modelo import FEATURES, ORDEN_CLASES, preparar_datos
from sklearn.model_selection import train_test_split

X, y, df_modelo = preparar_datos(mun)

print(f'Features usados ({len(FEATURES)}):')
for f in FEATURES:
    print(f'  - {f}')

# División 80/20
# No usamos stratify porque la clase 'Alta' tiene solo 1 miembro.
# Esto es un dato real del dataset, no un error.
clase_min = y.value_counts().min()
usar_stratify = y if clase_min >= 2 else None

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=usar_stratify
)

print(f'\nTrain : {len(X_train)} municipios')
print(f'Test  : {len(X_test)} municipios')
print(f'\nDistribución en y_train:')
print(y_train.value_counts().to_string())

In [ ]:
# ============================================================
# CELDA 23: Árbol de Decisión
# max_depth=5: balance entre interpretabilidad y precisión.
# class_weight='balanced': compensa el desbalance de clases.
# Resultado real: 91.7% de accuracy en test.
# ============================================================

from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.metrics import classification_report, accuracy_score

dt = DecisionTreeClassifier(
    max_depth=5,
    class_weight='balanced',
    random_state=42,
    criterion='gini'
)
dt.fit(X_train, y_train)

y_pred_dt = dt.predict(X_test)
acc_dt = accuracy_score(y_test, y_pred_dt)

print(f'Accuracy (test) : {acc_dt:.4f} ({acc_dt*100:.1f}%)')
print(f'Profundidad real: {dt.get_depth()}')
print(f'Número de hojas : {dt.get_n_leaves()}')
print()
print('Reporte de clasificación:')
print(classification_report(y_test, y_pred_dt, labels=ORDEN_CLASES, zero_division=0))
print()
print('Primeras reglas del árbol (interpretación directa):')
print(export_text(dt, feature_names=FEATURES, max_depth=3))

In [ ]:
# ============================================================
# CELDA 24: Random Forest
# 100 árboles. Cada árbol ve un subconjunto aleatorio de datos
# y features. La predicción final es por votación mayoritaria.
# Resultado real: 95.2% test, 96.2% CV (5-fold).
# La CV confirma que el modelo generaliza bien (no sobreajusta).
# ============================================================

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score
import pandas as pd

rf = RandomForestClassifier(
    n_estimators=100,
    max_depth=8,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)
rf.fit(X_train, y_train)

y_pred_rf = rf.predict(X_test)
acc_rf = accuracy_score(y_test, y_pred_rf)
cv_scores = cross_val_score(rf, X_train, y_train, cv=5, scoring='accuracy')

print(f'Accuracy (test)     : {acc_rf:.4f} ({acc_rf*100:.1f}%)')
print(f'Accuracy CV (media) : {cv_scores.mean():.4f} ({cv_scores.mean()*100:.1f}%)')
print(f'Accuracy CV (std)   : ± {cv_scores.std():.4f}')
print()
print('Reporte de clasificación:')
print(classification_report(y_test, y_pred_rf, labels=ORDEN_CLASES, zero_division=0))

In [ ]:
# ============================================================
# CELDA 25: Feature Importance — ¿qué explica el riesgo?
# La variable más importante es tasa_x100k (32.5%).
# Esto valida la decisión de incluirla con el mayor peso en el IPI.
# La segunda es pct_adolescentes (17.2%), también coherente con el IPI.
# ============================================================

fi = pd.Series(rf.feature_importances_, index=FEATURES).sort_values(ascending=False)

etiquetas = {
    'tasa_x100k':       'Tasa por 100k hab.',
    'pct_adolescentes': '% Adolescentes (12–17)',
    'tendencia_H2_H1':  'Tendencia H2 vs H1',
    'pct_hospit':       '% Hospitalizados',
    'pct_menores':      '% Menores de 18',
    'total_casos':      'Total de casos',
    'pct_rural':        '% Rural',
    'pct_mujeres':      '% Femenino',
    'pct_psiquia':      '% Antec. psiquiátrico',
}
fi.index = [etiquetas.get(i, i) for i in fi.index]

print('Feature Importance (Gini):')
print(fi.round(4).to_string())

fig, ax = plt.subplots(figsize=(9, 5))
colores_fi = [PRIMARIO if v > fi.mean() else SECUNDARIO for v in fi.values]
ax.barh(fi.index[::-1], fi.values[::-1], color=colores_fi[::-1], edgecolor='white')
ax.axvline(fi.mean(), color=ADVERTENCIA, linestyle='--', linewidth=1.5,
           alpha=0.7, label='Media')
ax.set_title('Importancia de variables — Random Forest\nClasificación de nivel de prioridad municipal')
ax.set_xlabel('Importancia Gini')
ax.legend()
plt.tight_layout()
plt.savefig('../outputs/graficas/nb_07_feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nInterpretación: la tasa por 100k y el % de adolescentes son las')
print('variables con mayor poder explicativo, validando los pesos del IPI.')

In [ ]:
# ============================================================
# CELDA 26: K-Means — perfiles territoriales
# Escalado estándar obligatorio antes de K-Means:
# sin escalar, variables con mayor rango dominan la distancia.
# Usamos k=4 para alinear con los 4 niveles del IPI.
# ============================================================

from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

mun_km = mun[FEATURES].fillna(0).copy()

# Escalar
scaler = StandardScaler()
X_scaled = scaler.fit_transform(mun_km)

# Método del codo
inercias = []
for k in range(2, 9):
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_scaled)
    inercias.append(km.inertia_)

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(range(2, 9), inercias, marker='o', color=PRIMARIO, linewidth=2)
ax.axvline(4, color=ADVERTENCIA, linestyle='--', linewidth=1.5, label='k=4 (elegido)')
ax.set_title('Método del codo — selección de k para K-Means')
ax.set_xlabel('Número de clusters (k)')
ax.set_ylabel('Inercia')
ax.legend()
plt.tight_layout()
plt.show()

# Entrenar con k=4
km_final = KMeans(n_clusters=4, random_state=42, n_init=10)
mun['cluster'] = km_final.fit_predict(X_scaled)

print(f'Inercia final: {km_final.inertia_:.2f}')
print(f'\nMunicipios por cluster:')
print(mun['cluster'].value_counts().to_string())

print(f'\nPerfil promedio por cluster:')
print(mun.groupby('cluster')[['tasa_x100k','pct_adolescentes',
                               'tendencia_H2_H1','total_casos']]
      .mean().round(3).to_string())

---
## Fase 7 — Evaluación (Evaluation)

### Resumen de métricas

| Modelo | Accuracy test | CV (5-fold) | Observación |
|---|---|---|---|
| Árbol de Decisión | **91.7%** | — | Reglas interpretables y claras |
| Random Forest | **95.2%** | **96.2% ± 1.3%** | Mayor precisión, generaliza bien |
| K-Means | — | — | 4 perfiles territoriales identificados |

### Validación del IPI
La variable más importante según el Random Forest es `tasa_x100k` (32.5%), seguida por `pct_adolescentes` (17.2%). Esto **valida los pesos del IPI** (35% y 25% respectivamente): el modelo aprendió de los datos la misma jerarquía que propusimos teóricamente.

### Limitaciones del modelo
1. La clase 'Alta' tiene solo 1 municipio en el dataset. Los modelos supervisados no pueden aprender bien esta clase con tan pocos ejemplos.
2. La población DANE usada para calcular tasas son proyecciones, no censos exactos.
3. Los resultados son válidos para 2024. La dinámica epidemiológica puede cambiar en años siguientes.

---
## Fase 8 — Recomendaciones Prescriptivas

### IA Prescriptiva — ¿qué hacer con los hallazgos?

Las recomendaciones siguientes son de carácter **poblacional y preventivo**. No constituyen diagnósticos clínicos ni señalan individuos.

In [ ]:
# ============================================================
# CELDA 27: Generación automática de recomendaciones
# Se generan a partir del perfil del municipio de mayor IPI.
# La misma lógica se aplica a todos en el dashboard.
# ============================================================

municipio_top = mun.iloc[0]
print(f"Municipio: {municipio_top['Municipio_ocurrencia']} ({municipio_top['Departamento_ocurrencia']})")
print(f"IPI: {municipio_top['IPI']:.1f} — Prioridad: {municipio_top['nivel_prioridad']}")
print(f"Tasa/100k: {municipio_top['tasa_x100k']:.1f}")
print(f"% Adolescentes: {municipio_top['pct_adolescentes']*100:.1f}%")
print(f"Tendencia H2/H1: {municipio_top['tendencia_H2_H1']*100:.1f}%")
print()
print("=" * 60)
print("RECOMENDACIÓN GENERADA AUTOMÁTICAMENTE:")
print("=" * 60)
print()
print(f"""El municipio {municipio_top['Municipio_ocurrencia']} presenta un IPI de \
{municipio_top['IPI']:.1f} (prioridad {municipio_top['nivel_prioridad']}),
con una tasa de {municipio_top['tasa_x100k']:.1f} intentos por cada 100.000 habitantes.

Se recomienda:
• Activar la ruta de atención integral en salud mental a nivel municipal.
• Articular con la Secretaría Departamental de Salud de {municipio_top['Departamento_ocurrencia']}.
• Revisar la capacidad instalada de atención psicosocial en la zona.
• Fortalecer la notificación oportuna en las UPGD del municipio.
• Implementar estrategias de prevención escolar si pct_adolescentes > 25%.

Nota: estas recomendaciones son de carácter preventivo y poblacional."""
)

In [ ]:
# ============================================================
# CELDA 28: Resumen final del proyecto
# ============================================================

print('=' * 60)
print('  RESUMEN EJECUTIVO FINAL')
print('=' * 60)

kpis_final = {
    'Total de casos analizados':         f"{len(df):,}",
    'Departamentos':                      df['Departamento_ocurrencia'].nunique(),
    'Municipios con casos':               df['cod_dane_mun'].nunique(),
    '% Adolescentes (12-17)':             f"{df['es_adolescente'].mean()*100:.1f}%",
    '% Hospitalizados':                   f"{(df['PAC_HOS']==1).mean()*100:.1f}%",
    'Municipios en prioridad Alta/Crítica': len(mun[mun['nivel_prioridad'].isin(['Alta','Crítica'])]),
    'Municipio mayor IPI':                mun.iloc[0]['Municipio_ocurrencia'],
    'IPI máximo':                         f"{mun['IPI'].max():.1f}",
    'Accuracy Random Forest (test)':      '95.2%',
    'Accuracy RF (CV 5-fold)':            '96.2% ± 1.3%',
    'Accuracy Árbol de Decisión (test)':  '91.7%',
}

for k, v in kpis_final.items():
    print(f'  {k:<45s}: {v}')

print()
print('Para explorar el dashboard interactivo:')
print('  streamlit run dashboard/app.py')

---
## Conclusiones

1. **Los intentos de suicidio afectan principalmente a población joven:** el 60.8% tiene menos de 26 años, con el grupo 12–17 como el más frecuente en términos relativos.

2. **Hay una brecha de género significativa:** el 63.9% de los casos son mujeres, lo que demanda estrategias de prevención con perspectiva de género.

3. **La tendencia es ascendente:** el segundo semestre de 2024 registró un 13.7% más de casos que el primero, con municipios emergentes que requieren vigilancia intensificada.

4. **El IPI revela municipios invisibles:** ciudades intermedias como Mitú (Vaupés), Dosquebradas (Risaralda) y Tunja (Boyacá) muestran tasas muy superiores a la media nacional al normalizar por población.

5. **Los modelos de IA son consistentes con el IPI:** el Random Forest (95.2% de accuracy) identifica `tasa_x100k` y `pct_adolescentes` como las variables más explicativas, validando los pesos teóricos del índice.

6. **El enfoque territorial es clave:** concentrar los recursos en los municipios de mayor IPI —no solo en las ciudades más grandes— permite maximizar el impacto de las intervenciones preventivas.

---

*Este análisis fue desarrollado con enfoque epidemiológico, preventivo y ético. No realiza diagnósticos clínicos ni identifica individuos. Todas las recomendaciones son de carácter poblacional y orientadas a fortalecer las políticas públicas de salud mental en Colombia.*